<a href="https://colab.research.google.com/github/kinjaljoshi/mech-interp-from-scratch/blob/master/Closed_form_least_squares_maths.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Closed-Form Least-Squares

Instead of using iterative gradient descent over multiple epochs, a linear probe can be solved analytically in "one shot" by treating classification as a multi-output linear regression problem.

Given an activation matrix $A$ of shape $(N, d)$ and a one-hot encoded target matrix $Y$ of shape $(N, C)$ where $C$ is the number of classes, we aim to find the optimal weight matrix $W$ of shape $(d, C)$.

---

## 1. Defining the Loss Function

We define our objective function as the minimization of the squared Frobenius norm of the residuals (the differences between our predictions $AW$ and one-hot targets $Y$):

$$L(W) = \|AW - Y\|_F^2$$

Using the matrix properties of the trace operator ($\text{Tr}$), where $\|M\|_F^2 = \text{Tr}(M^T M)$, we can expand this loss function:

$$L(W) = \text{Tr}\left( (AW - Y)^T (AW - Y) \right)$$
$$L(W) = \text{Tr}\left( (W^T A^T - Y^T)(AW - Y) \right)$$
$$L(W) = \text{Tr}(W^T A^T A W) - 2\text{Tr}(Y^T A W) + \text{Tr}(Y^T Y)$$

---

## 2. Matrix Calculus & The Normal Equations

To find the global minimum, we take the matrix derivative of the loss function with respect to $W$ and set it to zero:

$$\frac{\partial L}{\partial W} = 2 A^T A W - 2 A^T Y = 0$$

Dividing by 2 and isolating our variables yields the classic **Normal Equations**:

$$A^T A W = A^T Y$$

To isolate $W$, we multiply both sides from the left by the inverse matrix $(A^T A)^{-1}$:

$$W = (A^T A)^{-1} A^T Y$$

*Note: In practice, to prevent mathematical instability or issues with dead neurons/zero columns, we add a small ridge regularization term $\lambda I$, solving:*
$$W = (A^T A + \lambda I)^{-1} A^T Y$$

---

## 3. Numerical Example

Let's walk through a tiny toy setup with 3 samples, 2 activation features, and 2 target classes.

### The Inputs
Let our activation matrix $A$ and true class labels be:
* **Sample 1:** Features $[1, 2] \rightarrow$ Class 0
* **Sample 2:** Features $[2, 1] \rightarrow$ Class 0
* **Sample 3:** Features $[4, 5] \rightarrow$ Class 1

We convert our class integers into a one-hot encoded target matrix $Y$:

$$A = \begin{bmatrix} 1 & 2 \\ 2 & 1 \\ 4 & 5 \end{bmatrix}, \quad Y = \begin{bmatrix} 1 & 0 \\ 1 & 0 \\ 0 & 1 \end{bmatrix}$$

### Step-by-Step Resolution

#### Step A: Compute $A^T A$ (Feature Correlations)
$$A^T A = \begin{bmatrix} 1 & 2 & 4 \\ 2 & 1 & 5 \end{bmatrix} \begin{bmatrix} 1 & 2 \\ 2 & 1 \\ 4 & 5 \end{bmatrix} = \begin{bmatrix} 21 & 24 \\ 24 & 30 \end{bmatrix}$$

#### Step B: Compute the Inverse $(A^T A)^{-1}$
Using the 2D matrix inverse rule $\frac{1}{ad-bc}\begin{bmatrix} d & -b \\ -c & a \end{bmatrix}$ where $\text{det} = (21 \times 30) - (24 \times 24) = 54$:

$$(A^T A)^{-1} = \frac{1}{54} \begin{bmatrix} 30 & -24 \\ -24 & 21 \end{bmatrix} \approx \begin{bmatrix} 0.5556 & -0.4444 \\ -0.4444 & 0.3889 \end{bmatrix}$$

#### Step C: Compute $A^T Y$ (Class-Weighted Features)
$$A^T Y = \begin{bmatrix} 1 & 2 & 4 \\ 2 & 1 & 5 \end{bmatrix} \begin{bmatrix} 1 & 0 \\ 1 & 0 \\ 0 & 1 \end{bmatrix} = \begin{bmatrix} 3 & 4 \\ 3 & 5 \end{bmatrix}$$

#### Step D: Calculate the Final Weights $W$
$$W = (A^T A)^{-1} A^T Y = \begin{bmatrix} 0.5556 & -0.4444 \\ -0.4444 & 0.3889 \end{bmatrix} \begin{bmatrix} 3 & 4 \\ 3 & 5 \end{bmatrix} \approx \begin{bmatrix} 1/3 & 0 \\ -1/6 & 1/6 \end{bmatrix}$$

### Inference Verification
If we test a new sample skewed toward Class 1 with activations $A_{test} = \begin{bmatrix} 4 & 6 \end{bmatrix}$:

$$\text{Logits} = A_{test} W = \begin{bmatrix} 4 & 6 \end{bmatrix} \begin{bmatrix} 1/3 & 0 \\ -1/6 & 1/6 \end{bmatrix} = \begin{bmatrix} 0.333 & 1.0 \end{bmatrix}$$

Taking the index of the highest entry via `argmax` yields class **1**, matching our expectations perfectly in a single non-iterative operation.


In [ ]:
import torch

A = torch.tensor([[1, 2], [2, 1], [4, 5]])
Y = torch.tensor([[1, 0], [1, 0], [0, 1]])

#calculate transpose(A)@A and transpose(A)@Y
Ata = A.T @ A
Aty = A.T @ Y

#solve for W
W = torch.linalg.solve(Ata.float(), Aty.float())


In [ ]:
x_text = torch.tensor([[4, 6]]).float()
print(x_text @ W)
print(torch.argmax(x_text @ W))

tensor([[0.3333, 1.0000]])
tensor(1)
